# ⚛ EMN Doomsday Clock — Interactive Notebook

**Physics framing**: This notebook is your *particle detector control room*. Each cell is an instrument panel. We measure the four 'fields' driving EMN risk, combine them into a composite energy score, and map that onto a clock face.

### Architecture
```
Synthetic Market Data
        │
        ├─── Component A: Factor HHI (Quants' Hand)
        ├─── Component B: Corr / Beta Breakdown (Structural Hand)
        ├─── Component C: Crowding + Liquidity (Market Structure Hand)
        └─── Component D: Vol Regime + VIX + Surprise (External Hand)
                              │
                    Weighted Aggregation
                              │
                   Piecewise Linear Map
                              │
                   Minutes to Midnight ⏰
```

In [ ]:
# ── Setup: install deps if needed ───────────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg,
                           '--break-system-packages', '--ignore-installed', 'blinker', '-q'])

for pkg in ['plotly', 'pandas', 'numpy', 'scipy', 'statsmodels', 'dash', 'dash-bootstrap-components']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        install(pkg)

print('✅ Dependencies ready')

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from synthetic import generate_market_data
from component_a import normalize_component_a
from component_b import normalize_component_b
from component_c import normalize_component_c
from component_d import normalize_component_d
from aggregator import compute_clock, print_clock_report

print('✅ Modules loaded')

## 1. Generate Synthetic Market Data

Two years of daily data: factor returns, long/short book, VIX, IG spreads, crowding/liquidity scores.
A stress spike is injected in the last 60 days to test the clock's response.

In [ ]:
data = generate_market_data(n_days=504, seed=42)

print('Date range:', data['dates'][0].date(), '→', data['dates'][-1].date())
print('Factor exposures (current snapshot):')
print(data['factor_exposures'].to_string())

In [ ]:
# ── Visualise input data ─────────────────────────────────────────────────────
fig = make_subplots(rows=3, cols=2,
    subplot_titles=['Portfolio Cum. Return', 'VIX',
                    'Long vs Short Book', 'IG Spread (bps)',
                    'Crowding Score (0-10)', 'Liquidity Score (0-10)'],
    vertical_spacing=0.12, horizontal_spacing=0.08)

cum = (1 + data['portfolio_returns']).cumprod()
fig.add_trace(go.Scatter(x=data['dates'], y=cum, name='Portfolio', line=dict(color='#00bcd4')), row=1, col=1)
fig.add_trace(go.Scatter(x=data['dates'], y=data['vix'], name='VIX', line=dict(color='#ff9800')), row=1, col=2)

long_cum = (1 + data['long_returns']).cumprod()
short_cum = (1 + data['short_returns']).cumprod()
fig.add_trace(go.Scatter(x=data['dates'], y=long_cum, name='Long', line=dict(color='#4caf50')), row=2, col=1)
fig.add_trace(go.Scatter(x=data['dates'], y=short_cum, name='Short', line=dict(color='#f44336')), row=2, col=1)
fig.add_trace(go.Scatter(x=data['dates'], y=data['ig_spread'], name='IG Spread', line=dict(color='#ab47bc')), row=2, col=2)
fig.add_trace(go.Scatter(x=data['dates'], y=data['crowding_scores'], name='Crowding', line=dict(color='#ff7043')), row=3, col=1)
fig.add_trace(go.Scatter(x=data['dates'], y=data['liquidity_scores'], name='Liquidity', line=dict(color='#26c6da')), row=3, col=2)

fig.update_layout(height=650, title='Market Data Overview', template='plotly_dark', showlegend=True)
fig.show()

## 2. Component A: Factor Exposure Risk

**Metric**: Herfindahl-Hirschman Index of squared factor loadings.
$$\text{HHI} = \sum_{i} e_i^2$$
This is the L² norm of the exposure vector squared — a measure of *anisotropy* in factor space.

In [ ]:
res_a = normalize_component_a(data['factor_exposures'])

print(f"Raw HHI       : {res_a['raw_hhi']:.4f}")
print(f"Normalized A  : {res_a['normalized_score']:.2f} / 10")
print(f"Max exposure  : {res_a['max_exposure_factor']} = {res_a['max_exposure_value']:.4f}")

# Bar chart: factor exposures + contributions
factors = list(res_a['factor_exposures'].keys())
exposures = list(res_a['factor_exposures'].values())
contributions = [res_a['factor_contributions'][f] for f in factors]

fig = go.Figure()
fig.add_trace(go.Bar(x=factors, y=exposures, name='Factor Exposure',
                     marker_color=['#4caf50' if e >= 0 else '#f44336' for e in exposures]))
fig.add_trace(go.Bar(x=factors, y=contributions, name='HHI Contribution',
                     marker_color='#ff9800', opacity=0.7))
fig.update_layout(title=f'Component A: Factor Exposures  (Score: {res_a["normalized_score"]:.1f}/10)',
                  template='plotly_dark', barmode='group', height=350)
fig.show()

## 3. Component B: Correlation Breakdown Risk

The **L/S correlation** should sit around –0.6. When it drifts toward 0 or positive, the portfolio is no longer neutral — it's a directional bet. The **rolling beta** to the market quantifies the same effect from a different angle.

In [ ]:
res_b = normalize_component_b(
    data['long_returns'], data['short_returns'],
    data['portfolio_returns'], data['market_returns']
)

print(f"Current L/S corr   : {res_b['current_long_short_corr']:.4f}")
print(f"1yr baseline corr  : {res_b['baseline_corr_1yr']:.4f}")
print(f"Corr deviation     : {res_b['corr_deviation']:.4f}")
print(f"Current beta       : {res_b['current_beta']:.4f}")
print(f"Score (corr)       : {res_b['score_corr_breakdown']:.2f}")
print(f"Score (beta)       : {res_b['score_beta_drift']:.2f}")
print(f"Normalized B       : {res_b['normalized_score']:.2f} / 10")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['Rolling 60d L/S Correlation', 'Rolling 60d Portfolio Beta to Market'])

corr_s = res_b['_corr_series']
beta_s = res_b['_beta_series']

fig.add_trace(go.Scatter(x=data['dates'], y=corr_s, name='L/S Corr',
                         line=dict(color='#e91e63', width=1.5)), row=1, col=1)
fig.add_hline(y=res_b['baseline_corr_1yr'], line_dash='dash', line_color='#888888',
              annotation_text='1yr baseline', row=1, col=1)
fig.add_hline(y=0, line_dash='dot', line_color='#555', row=1, col=1)

fig.add_trace(go.Scatter(x=data['dates'], y=beta_s, name='Beta',
                         line=dict(color='#ff9800', width=1.5)), row=2, col=1)
fig.add_hline(y=0, line_dash='dot', line_color='#555', row=2, col=1)

fig.update_layout(height=450, template='plotly_dark',
    title=f'Component B: Correlation Breakdown  (Score: {res_b["normalized_score"]:.1f}/10)')
fig.show()

## 4. Component C: Crowding & Liquidity Risk

Think of crowding as **entropy reduction** — everyone herding into the same positions. When combined with poor liquidity (narrow escape valve), small perturbations trigger an avalanche.

In [ ]:
res_c = normalize_component_c(
    crowding_score=float(data['crowding_scores'].iloc[-1]),
    liquidity_score=float(data['liquidity_scores'].iloc[-1]),
    vix_current=float(data['vix'].iloc[-1]),
    ig_spread_current=float(data['ig_spread'].iloc[-1]),
)

print(f"Portfolio crowding  : {res_c['portfolio_crowding_raw']:.3f}")
print(f"Portfolio liquidity : {res_c['portfolio_liquidity_raw']:.3f}")
print(f"VIX current         : {res_c['vix_current']:.1f}")
print(f"IG spread           : {res_c['ig_spread_current']:.0f} bps")
print(f"Score crowding      : {res_c['score_crowding']:.2f}")
print(f"Score portfolio liq : {res_c['score_portfolio_liquidity']:.2f}")
print(f"Score market liq    : {res_c['score_market_liquidity']:.2f}")
print(f"Normalized C        : {res_c['normalized_score']:.2f} / 10")

# Waterfall of sub-scores
sub_labels = ['Crowding (35%)', 'Port. Liquidity (35%)', 'Market Liq. (30%)', 'Total']
sub_scores = [
    0.35 * res_c['score_crowding'],
    0.35 * res_c['score_portfolio_liquidity'],
    0.30 * res_c['score_market_liquidity'],
    res_c['normalized_score']
]
colors = ['#ff7043', '#26c6da', '#ab47bc', '#ff9800']

fig = go.Figure(go.Bar(
    x=sub_labels, y=sub_scores,
    marker_color=colors,
    text=[f'{v:.2f}' for v in sub_scores], textposition='outside'
))
fig.update_layout(title=f'Component C: Sub-score Breakdown  (Score: {res_c["normalized_score"]:.1f}/10)',
                  template='plotly_dark', yaxis_range=[0, 12], height=320)
fig.show()

## 5. Component D: Macro Shock Risk

The **external thermal bath** of the system. Realized vol regime detection tells us which 'temperature' the market is running at. VIX percentile ranks the *implied* fear. The economic surprise proxy flags sudden regime shifts.

In [ ]:
res_d = normalize_component_d(
    data['portfolio_returns'], data['market_returns'], data['vix']
)

print(f"Volatility regime   : {res_d['regime']}")
print(f"Annualised vol      : {res_d['current_annualized_vol']:.2%}")
print(f"Vol percentile      : {res_d['vol_percentile']:.2%}")
print(f"Regime score        : {res_d['regime_score']:.2f}")
print(f"VIX                 : {res_d['current_vix']:.2f}")
print(f"VIX 1yr percentile  : {res_d['vix_percentile']:.2%}")
print(f"VIX score           : {res_d['vix_score']:.2f}")
print(f"Surprise proxy      : {res_d['surprise_proxy']:.5f}")
print(f"Surprise score      : {res_d['surprise_score']:.2f}")
print(f"Normalized D        : {res_d['normalized_score']:.2f} / 10")

# Rolling vol + VIX overlay
rv = res_d['_rv_series']
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['Annualised Realised Volatility (21d)', 'VIX Level'])

fig.add_trace(go.Scatter(x=rv.index, y=rv * 100, name='Realised Vol %',
                         line=dict(color='#4fc3f7')), row=1, col=1)
fig.add_trace(go.Scatter(x=data['dates'], y=data['vix'], name='VIX',
                         line=dict(color='#ff9800')), row=2, col=1)

fig.update_layout(height=400, template='plotly_dark',
    title=f'Component D: Macro Shock  (Score: {res_d["normalized_score"]:.1f}/10)')
fig.show()

## 6. Aggregation → Clock Reading

**Composite score** = weighted dot product of 4 normalised components.
$$\text{Score} = 0.35 A + 0.30 B + 0.25 C + 0.10 D$$
Mapped via piecewise linear function to *minutes to midnight*.

In [ ]:
reading = compute_clock(
    res_a['normalized_score'], res_b['normalized_score'],
    res_c['normalized_score'], res_d['normalized_score'],
    res_a, res_b, res_c, res_d,
)
print_clock_report(reading)

## 7. Clock Face Visualisation

In [ ]:
# Re-use the clock figure from the dashboard module
import importlib.util, types, os

# Load only the figure-making function (avoid launching the Dash server)
_dashboard_path = os.path.abspath('dashboard.py')
dashboard_mod = types.ModuleType('dashboard')
dashboard_mod.__file__ = _dashboard_path
exec(open(_dashboard_path).read().split("if __name__ == '__main__'")[0], dashboard_mod.__dict__)

clock_fig = dashboard_mod.make_clock_figure(reading)
clock_fig.show()

## 8. Rolling Clock History

Compute the clock reading for every day in the last year. This is your risk audit trail — you should be able to see the clock advance during the injected stress spike.

In [ ]:
LOOKBACK = 252  # compute rolling clock for last 252 trading days
WINDOW = 120    # minimum data needed for rolling calcs

history = []
print(f'Computing rolling clock ({LOOKBACK} days)...', end=' ')

for i in range(WINDOW, len(data['dates'])):
    # Slice data up to day i
    sl = slice(0, i+1)
    d = {
        k: v.iloc[sl] if hasattr(v, 'iloc') else v
        for k, v in data.items() if k != 'dates'
    }
    d['dates'] = data['dates']

    try:
        a = normalize_component_a(data['factor_exposures'])
        b = normalize_component_b(d['long_returns'], d['short_returns'],
                                   d['portfolio_returns'], d['market_returns'])
        c = normalize_component_c(
            float(d['crowding_scores'].iloc[-1]),
            float(d['liquidity_scores'].iloc[-1]),
            float(d['vix'].iloc[-1]),
            float(d['ig_spread'].iloc[-1]),
        )
        dd = normalize_component_d(d['portfolio_returns'], d['market_returns'], d['vix'])
        r = compute_clock(a['normalized_score'], b['normalized_score'],
                          c['normalized_score'], dd['normalized_score'],
                          a, b, c, dd)
        history.append({
            'date': data['dates'][i],
            'minutes': r.minutes_to_midnight,
            'score': r.raw_score,
            'zone': r.zone,
            'A': r.component_scores['A'],
            'B': r.component_scores['B'],
            'C': r.component_scores['C'],
            'D': r.component_scores['D'],
        })
    except Exception:
        pass

hist_df = pd.DataFrame(history)
print(f'Done. {len(hist_df)} readings.')
print(hist_df.tail(5).to_string(index=False))

In [ ]:
# ── Plot rolling clock history ──────────────────────────────────────────────
ZONE_BANDS = [
    (21, 23, 'rgba(0,200,83,0.08)',  'Safe'),
    (11, 20, 'rgba(255,214,0,0.08)', 'Normal'),
    (6,  10, 'rgba(255,109,0,0.08)', 'Elevated'),
    (0,   5, 'rgba(213,0,0,0.15)',   'Critical'),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['Minutes to Midnight (lower = more dangerous)',
                    'Component Scores (A/B/C/D)'],
    vertical_spacing=0.10)

for lo, hi, color, label in ZONE_BANDS:
    fig.add_hrect(y0=lo, y1=hi, fillcolor=color, line_width=0,
                  annotation_text=label, annotation_position='right',
                  row=1, col=1)

fig.add_trace(go.Scatter(
    x=hist_df['date'], y=hist_df['minutes'],
    fill='tozeroy', fillcolor='rgba(255,68,68,0.10)',
    line=dict(color='#ff4444', width=2),
    name='Minutes to Midnight'), row=1, col=1)

comp_colors = {'A': '#ff9800', 'B': '#e91e63', 'C': '#00bcd4', 'D': '#69f0ae'}
for comp, col in comp_colors.items():
    fig.add_trace(go.Scatter(
        x=hist_df['date'], y=hist_df[comp],
        name=f'Component {comp}', line=dict(color=col, width=1.2)), row=2, col=1)

fig.update_layout(height=550, template='plotly_dark',
    title='Rolling EMN Doomsday Clock History')
fig.update_yaxes(range=[0, 24], row=1, col=1)
fig.update_yaxes(range=[0, 11], row=2, col=1)
fig.show()

## 9. Sensitivity Analysis: Weight Tuning

How sensitive is the clock reading to the weight choices? This is your **calibration cell** — adjust weights and instantly see the clock move.

In [ ]:
from aggregator import compute_clock, DEFAULT_WEIGHTS

# ── Sweep weight_A from 0.10 to 0.60 (keeping others proportional) ──────────
weight_A_range = np.linspace(0.10, 0.60, 30)
results = []

for wA in weight_A_range:
    remaining = 1.0 - wA
    wB = remaining * (0.30 / 0.65)
    wC = remaining * (0.25 / 0.65)
    wD = remaining * (0.10 / 0.65)
    w = {'A': wA, 'B': wB, 'C': wC, 'D': wD}
    r = compute_clock(
        res_a['normalized_score'], res_b['normalized_score'],
        res_c['normalized_score'], res_d['normalized_score'],
        res_a, res_b, res_c, res_d, weights=w
    )
    results.append({'weight_A': wA, 'minutes': r.minutes_to_midnight, 'score': r.raw_score})

sweep_df = pd.DataFrame(results)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Minutes to Midnight vs Weight_A', 'Raw Score vs Weight_A'])

fig.add_trace(go.Scatter(x=sweep_df['weight_A'], y=sweep_df['minutes'],
                          line=dict(color='#ff4444', width=2), name='Minutes'), row=1, col=1)
fig.add_trace(go.Scatter(x=sweep_df['weight_A'], y=sweep_df['score'],
                          line=dict(color='#ff9800', width=2), name='Score'), row=1, col=2)

# Mark current default weight
default_wA = DEFAULT_WEIGHTS['A']
for row in [1, 2]:
    fig.add_vline(x=default_wA, line_dash='dash', line_color='#00ff88',
                  annotation_text=f'default={default_wA}', row=1, col=row)

fig.update_layout(height=350, template='plotly_dark',
    title='Sensitivity: Factor Exposure Weight (A)')
fig.show()

print(f"\nCurrent default weights: {DEFAULT_WEIGHTS}")
print(f"Clock reading at defaults: {reading.minutes_to_midnight:.1f} min  [{reading.zone}]")

## 10. Launch Dashboard (optional)

Run the cell below to start the full Dash dashboard on `http://127.0.0.1:8050`.
The dashboard auto-refreshes every 60 seconds.

> **Note**: This will block the notebook kernel. Interrupt the kernel to stop.

In [ ]:
# Uncomment to launch dashboard
# import subprocess
# subprocess.Popen(['python', 'doomsday_clock/dashboard.py'])
# print('Dashboard launched at http://127.0.0.1:8050')
print('Uncomment the lines above to launch the dashboard.')

---
## Summary

| Component | Score | Weight | Contribution |
|-----------|-------|--------|--------------|
| A · Factor Exposure | — | 35% | — |
| B · Corr Breakdown  | — | 30% | — |
| C · Crowding & Liq  | — | 25% | — |
| D · Macro Shock     | — | 10% | — |
| **Composite**       | — |  —  | — |

*Fill in from `print_clock_report(reading)` above.*

**Next steps for production:**
- Replace `data/synthetic.py` with Bloomberg/FactSet API connectors
- Calibrate `HISTORICAL_MAX_*` constants on actual crisis data (Aug 2007, Mar 2020)
- Wire the dashboard to a live database and add auto-refresh
- Add alerting (email/Slack) when zone crosses Elevated or Critical